<a href="https://colab.research.google.com/github/Quantum00000/Kaggle_Compititions/blob/main/Kaggle_CIFAR_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torchvision import transforms


Data import from kaggle

In [ ]:
import os
os.environ['KAGGLE_API_TOKEN'] = 'KGAT_5266d2b9f41bb08f77787f1fc1d7ce93'

In [ ]:
mkdir -p ~/.kaggle && echo KGAT_5266d2b9f41bb08f77787f1fc1d7ce93 > ~/.kaggle/access_token && chmod 600 ~/.kaggle/access_token

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.competition_download('cifar-10neucalssification')

print("Path to competition files:", path)

In [ ]:
print(os.listdir(path))

In [ ]:
train_labels=pd.read_csv(os.path.join(path,"train_labels.csv"))
train_labels.head()

Extracting img and label number

In [ ]:
import cv2

x_train=[]
y_train=[]

for _,row in train_labels.iterrows():

    img_path=os.path.join(path,"train",row["filename"])

    img=cv2.imread(img_path)
    img=cv2.cvtColor(img,cv2.COLOR_BGR2RGB)

    x_train.append(img)
    y_train.append(row['label'])

x_train=np.array(x_train)
y_train=np.array(y_train)


img=x_train, label=y_train

In [ ]:
y_train=torch.tensor(y_train)
x_train=torch.tensor(x_train,dtype=torch.float32) / 255.0
x_train=x_train.permute(0 ,3 , 1, 2)

In [ ]:
print(x_train[0].shape)

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset=TensorDataset(x_train,y_train)
train_loader=DataLoader(train_dataset,
                        batch_size=128,
                        shuffle=True,
                        num_workers=2,
                        drop_last=True)

Classes for label

In [ ]:
Classes={
    0:"airplane",
    1:"automobile",
    2:"bird",
    3:"cat",
    4:"deer",
    5:"dog",
    6:"frog",
    7:"horse",
    8:"ship",
    9:"truck"
}

In [ ]:
img,label=train_dataset[0]

In [ ]:
print(img.shape)
# Normalize the image data to the range [0, 1] for proper display
plt.imshow(img.permute(1, 2, 0))
print("Label:", Classes[label.item()])

In [ ]:
train_feature_batch,train_label_batch=next(iter(train_loader))

In [ ]:
Classes[train_label_batch[0].item()]

In [ ]:
plt.imshow(train_feature_batch[0].permute(1, 2, 0))
plt.show()

Model Training

In [ ]:
class CIFARMODEL_01(nn.Module):

    def __init__(self,input_shape:int,hidden_unit:int,output_shape):
      super().__init__()
      self.block1=nn.Sequential (
          nn.Conv2d(in_channels=input_shape,
                    out_channels=hidden_unit,
                    kernel_size=3,
                    padding=1,
                    stride=1),
          nn.ReLU(),
          nn.Conv2d(in_channels=hidden_unit,
                    out_channels=hidden_unit,
                    kernel_size=3,
                    stride=1,
                    padding=1),
          nn.ReLU(),
          nn.MaxPool2d(kernel_size=2,
                       stride=2)
      )
      self.block2=nn.Sequential(
          nn.Conv2d(in_channels=hidden_unit,
                    out_channels=hidden_unit,
                    kernel_size=3,
                    stride=1,
                    padding=1),
          nn.ReLU(),
          nn.Conv2d(in_channels=hidden_unit,
                    out_channels=hidden_unit,
                    kernel_size=3,
                    stride=1,
                    padding=1),
          nn.ReLU(),
          nn.MaxPool2d(kernel_size=2,
                       stride=2)
      )
      self.Classifier=nn.Sequential(
          nn.Flatten(),
          nn.Linear(in_features=hidden_unit*8*8,
                    out_features=output_shape)
      )
    def forward(self,x:torch.Tensor):
      #x = x.permute(0, 3, 1, 2) # Permute dimensions for Conv2d
      x=self.block1(x)
      x=self.block2(x)
      x=self.Classifier(x)
      return x

# Model training and Model evaluation


In [ ]:
model01=CIFARMODEL_01(
    input_shape=3, # Changed input_shape to 3 for RGB channels
    hidden_unit=32,
    output_shape=len(Classes)
)
model01

Optimizer

In [ ]:
optimizer=torch.optim.SGD(params=model01.parameters(),
                          lr=0.01,
                          weight_decay=1e-4,
                          momentum=0.9)

Loss Function

In [ ]:
Loss_fn=nn.CrossEntropyLoss()

Accuracy function

In [ ]:
def accuracy_fn(y_true,y_pred):
  correct=torch.eq(y_true,y_pred).sum().item()
  acc=correct/len(y_pred)
  return acc

Model training

In [ ]:
def Train_model(
    model:torch.nn.Module,
    data_loader:torch.utils.data.DataLoader,
    loss_fn:torch.nn.Module,
    optimizer:torch.optim.Optimizer,
    accuracy_fn,
    device:torch.device=torch.device
):

  torch.manual_seed(42)

  epoches=30

  for epoch in range(epoches):
    train_loss,train_acc=0,0
    model.to(device)
    for batch,((X,y)) in enumerate(data_loader):
      X,y=X.to(device),y.to(device)
      y_pred=model(X)
      loss=loss_fn(y_pred,y)
      train_loss+=loss.item()
      train_acc+=accuracy_fn(y,y_pred.argmax(dim=1))
      optimizer.zero_grad()
      loss.backward()
      optimizer.step()

    train_loss/=len(data_loader)
    train_acc/=len(data_loader)
    print(f"Loss:{train_loss:.5f}|Accuracy:{train_acc:.5f}")


In [ ]:
Train_model(
    model=model01,
    data_loader=train_loader,
    loss_fn=Loss_fn,
    optimizer=optimizer,
    accuracy_fn=accuracy_fn,
    device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
)

In [ ]:
print(next(model01.parameters()).device)

# validation

In [ ]:
val_label=pd.read_csv(os.path.join(path,"val_labels.csv"))
val_label.head()
val_label.shape

In [ ]:
x_val=[]
y_val=[]

for _,row in val_label.iterrows():
  img_path=os.path.join(path,"val",row["filename"])
  img=cv2.imread(img_path)
  img=cv2.cvtColor(img,cv2.COLOR_BGR2RGB)
  x_val.append(img)
  y_val.append(row["label"])
x_val=np.array(x_val)
y_val=np.array(y_val)


In [ ]:
x_val=torch.tensor(x_val,dtype=torch.float32) /255.0
x_val=x_val.permute(0, 3, 1, 2)
y_val=torch.tensor(y_val)

In [ ]:
val_dataset=TensorDataset(x_val,y_val)
val_dataloader=DataLoader(val_dataset,
                          shuffle=True,
                          batch_size=128,
                          num_workers=2,
                          drop_last=True)

Model Evaluation

In [ ]:
def Test_Model(
    model:torch.nn.Module,
    data_loader:torch.utils.data.DataLoader,
    loss_fn:torch.nn.Module,
    accuracy_fn,
    device:torch.device=torch.device):

  test_loss,test_acc=0,0
  model01.to(device)
  model01.eval()
  with torch.inference_mode():
    for X,y in data_loader:
      X,y=X.to(device),y.to(device)
      test_pred=model01(X)
      loss=loss_fn(test_pred,y)
      test_loss+=loss.item()
      test_acc+=accuracy_fn(y,test_pred.argmax(dim=1))
    test_loss/=len(data_loader)
    test_acc/=len(data_loader)
    print(f"Test Loss:{test_loss:.5f}|Test Accuracy:{test_acc:.5f}")



In [ ]:
Test_Model(
    model=model01,
    data_loader=val_dataloader,
    loss_fn=Loss_fn,
    accuracy_fn=accuracy_fn,
    device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
)

# Test Set

In [ ]:
test_path=os.path.join(path,"test")

In [ ]:
x_test=[]
for filename in os.listdir(test_path):
  test_img_path=os.path.join(test_path,filename)
  test_img=cv2.imread(test_img_path)
  test_img=cv2.cvtColor(img,cv2.COLOR_BGR2RGB)
  x_test.append(img)

In [ ]:
x_test=np.array(x_test)
x_test=torch.tensor(x_test,dtype=torch.float32) / 255.0
x_test=x_test.permute(0, 3, 1, 2)

In [ ]:
device="cuda" if torch.cuda.is_available() else "cpu"
x_test=x_test.to(device)
model01.to(device)
model01.eval()
with torch.inference_mode():
  test_pred=model01(x_test)


In [ ]:
test_pred

In [ ]:
submit=pd.DataFrame({
    "filename":os.listdir(test_path),
    "label":test_pred.to("cpu").argmax(dim=1)
})
submit.to_csv('submission.csv',index=False)